In [9]:
def make_prompt(example):
    """
    Create a prompt for the model.
    
    Args:
        example: The input example.
    
    Returns:
        dict: The prompt for the model.
    """
    return {
                "prompt": f"""### Instruction:\nYou are an expert in legal language and its interpretation. 
                Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. 
                The simplified version should be accessible to individuals without a legal background, using clear and concise language. 
                ### Input:\n{example}
                """
            }

In [5]:
import os
os.chdir("/home/danish/Legal-Simplification-mrabbani")
os.environ["PYTHONPATH"] = f"SFT_trainer"
import json
import time
from SFT_trainer.evaluation import Evaluation
import logging
from datasets import load_dataset,load_from_disk
from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 05-21 18:27:42 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 05-21 18:27:42 [__init__.py:239] Automatically detected platform cuda.


2025-05-21 18:27:46,845	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [15]:
model_path = "drd01/llama_3.2_1B_SIMPLE-LAW" 
 
model, tokenizer = FastLanguageModel.from_pretrained(
            model_path,
            max_seq_length=8192,  # Choose any for long context!
            load_in_4bit=True,  # 4 bit quantization to reduce memory
            load_in_8bit=False,  # [NEW!] A bit more accurate, uses 2x memory
            # full_finetuning = True, # [NEW!] We have full finetuning now!
            # token = "hf_...", # use one if using gated models
        )

==((====))==  Unsloth 2025.4.7: Fast Llama patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    NVIDIA GeForce RTX 2080 Ti. Num GPUs = 1. Max memory: 10.569 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.4.7 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [29]:
def generate(model, tokenizer, example):
    prompt  = "{}"
    inputs = tokenizer([prompt.format(example['prompt']),], return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    input_length = input_ids.shape[1]
    simplified_text = model.generate(**inputs, 
                                        max_new_tokens = 8192, 
                                        eos_token_id=tokenizer.eos_token_id, 
                                        temperature=0.7,
                                        top_k=50,
                                        top_p=0.9,
                                    repetition_penalty=1.2,
                                    no_repeat_ngram_size=3)
    simplified_text = simplified_text[:, input_length:]
    simplified_text = tokenizer.decode(simplified_text[0], skip_special_tokens=True).strip()
    print(simplified_text)

In [20]:
legal_text  = "The lessee shall indemnify the lessor against all liabilities arising from the use of the premises."

In [32]:
example = {"prompt":make_prompt(legal_text)}
generate(model, tokenizer, example)

']
                    """
Indemnification Obligation for Use of Premises
The lessee must pay any costs or expenses incurred by the lessors as a result of their use of these properties.
                     """
